# MedSAM3 cancer-cell instance segmentation and counting on Kaggle TPU

This notebook:

1. Installs the `feature/cell-instance-counting` branch of `Locutusque/MedSAM3`.
2. Installs PyTorch/XLA and AutoXLA for TPU inference.
3. Downloads the MedSAM3 LoRA checkpoint from Hugging Face.
4. Runs the text prompt **“cancer cell”** on one image.
5. Removes small objects, assigns a distinct color to every instance, and reports the count.

## Kaggle settings

Before running the notebook:

- Open **Settings → Accelerator → TPU VM v5e-8**.
- Turn **Internet** on.
- Add a Kaggle secret named `HF_TOKEN` if the Hugging Face model requires authentication.
- Add your image as a Kaggle Dataset, or upload it into `/kaggle/working`.

The first TPU inference can be slow because XLA compiles the graph. Later runs with the same tensor shapes should be faster.


In [ ]:
# Confirm that this is a Kaggle TPU runtime.
import os
import platform
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Kaggle:", os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "not detected"))
print("TPU_NAME:", os.environ.get("TPU_NAME", "not set"))
print("PJRT_DEVICE:", os.environ.get("PJRT_DEVICE", "not set"))

if not os.environ.get("TPU_NAME") and os.environ.get("PJRT_DEVICE", "").upper() != "TPU":
    print(
        "\nWARNING: A TPU runtime was not detected. "
        "In Kaggle, select Settings → Accelerator → TPU VM v5e-8."
    )


## 1. Install MedSAM3, PyTorch/XLA, and AutoXLA

This cell follows the TPU versions and package sources used by the repository. Package installation may take several minutes.

After the installation completes, **restart the Kaggle session once** if the next import cell reports a PyTorch or `torch_xla` binary mismatch. Then continue from the import-verification cell.


In [ ]:
%%bash
set -euxo pipefail

cd /kaggle/working

# Clone or refresh the branch containing the counting scripts.
if [ -d MedSAM3/.git ]; then
  cd MedSAM3
  git fetch origin feature/cell-instance-counting
  git checkout feature/cell-instance-counting
  git pull --ff-only origin feature/cell-instance-counting
  cd ..
else
  git clone --branch feature/cell-instance-counting \
    https://github.com/Locutusque/MedSAM3.git
fi

python -m pip install --upgrade pip setuptools wheel

# PyTorch/XLA TPU wheels used by this repository.
python -m pip install "torch~=2.8.0"
python -m pip install "torch_xla[tpu]~=2.8.0" \
  --find-links=https://storage.googleapis.com/libtpu-releases/index.html \
  --find-links=https://storage.googleapis.com/libtpu-wheels/index.html

python -m pip install "torch_xla[pallas]" \
  --find-links=https://storage.googleapis.com/jax-releases/jax_nightly_releases.html \
  --find-links=https://storage.googleapis.com/jax-releases/jaxlib_nightly_releases.html

# AutoXLA branch currently used by MedSAM3's TPU integration.
if [ -d autoxla/.git ]; then
  cd autoxla
  git fetch origin claude/image-segmentation-quantization-l5h9x9
  git checkout claude/image-segmentation-quantization-l5h9x9
  git pull --ff-only origin claude/image-segmentation-quantization-l5h9x9
  cd ..
else
  git clone --branch claude/image-segmentation-quantization-l5h9x9 \
    https://github.com/Locutusque/autoxla.git
fi

python -m pip install -e /kaggle/working/autoxla
python -m pip install -e /kaggle/working/MedSAM3
python -m pip install huggingface-hub


In [ ]:
# Verify TPU imports before loading the large model.
import os
os.environ.setdefault("PJRT_DEVICE", "TPU")

import torch
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr

import sys
sys.path.insert(0, "/kaggle/working/MedSAM3")
import tpu_utils

print("torch:", torch.__version__)
print("torch_xla:", getattr(torch_xla, "__version__", "unknown"))
print("AutoXLA importable:", tpu_utils.AUTOXLA_AVAILABLE)
print("Global TPU devices:", xr.global_runtime_device_count())

device = tpu_utils.get_xla_device()
print("Selected XLA device:", device)


## 2. Authenticate with Hugging Face and download the LoRA weights

The notebook first looks for the Kaggle secret `HF_TOKEN`. If it is absent, public downloads are attempted without authentication.


In [ ]:
from pathlib import Path
from huggingface_hub import login, snapshot_download

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    print("HF_TOKEN was not loaded; trying an unauthenticated download.")
    print("Reason:", exc)

if hf_token:
    login(token=hf_token, add_to_git_credential=False)

weights_dir = Path(
    snapshot_download(
        repo_id="lal-Joey/MedSAM3_v1",
        token=hf_token,
        local_dir="/kaggle/working/medsam3_weights",
    )
)

weight_candidates = sorted(
    list(weights_dir.rglob("*.pt"))
    + list(weights_dir.rglob("*.pth"))
    + list(weights_dir.rglob("*.bin"))
    + list(weights_dir.rglob("*.safetensors"))
)

print("Downloaded to:", weights_dir)
print("Weight candidates:")
for path in weight_candidates:
    print(" -", path)

if not weight_candidates:
    raise FileNotFoundError(
        "No checkpoint file was found in the Hugging Face snapshot. "
        "Inspect the downloaded directory and set WEIGHTS_PATH manually."
    )

# Prefer a filename that clearly looks like the LoRA checkpoint.
preferred = [
    path for path in weight_candidates
    if "lora" in path.name.lower() or "best" in path.name.lower()
]
WEIGHTS_PATH = preferred[0] if preferred else weight_candidates[0]
print("\nSelected checkpoint:", WEIGHTS_PATH)


## 3. Select the input image and inference settings

Set `IMAGE_PATH` to one image in `/kaggle/input/...` or `/kaggle/working/...`.

For densely packed cells, the most important parameters are:

- `THRESHOLD`: lower values increase recall but may add false positives.
- `MIN_AREA`: masks smaller than this pixel count are removed.
- `MASK_NMS_IOU`: duplicate-mask suppression threshold.
- `RESOLUTION`: must remain fixed between runs to maximize XLA compilation reuse.


In [ ]:
from pathlib import Path

# EDIT THIS PATH.
IMAGE_PATH = Path("/kaggle/input/YOUR-DATASET/YOUR_IMAGE.png")

PROMPT = "cancer cell"
THRESHOLD = 0.45
NMS_IOU = 0.50
MASK_NMS_IOU = 0.65
MIN_AREA = 80
MAX_HOLE_AREA = 32
RESOLUTION = 1008
OUTPUT_DIR = Path("/kaggle/working/cell_count_output")

CONFIG_PATH = Path("/kaggle/working/MedSAM3/configs/full_lora_config.yaml")

if not IMAGE_PATH.exists():
    raise FileNotFoundError(
        f"Input image does not exist: {IMAGE_PATH}\n"
        "Edit IMAGE_PATH so it points to your Kaggle Dataset or uploaded image."
    )
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config does not exist: {CONFIG_PATH}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Image:", IMAGE_PATH)
print("Weights:", WEIGHTS_PATH)
print("Config:", CONFIG_PATH)
print("Output:", OUTPUT_DIR)


## 4. Run prompted MedSAM3 inference on the TPU

The TPU runs the model forward pass through the repository’s `torch_xla` and AutoXLA integration. Mask cleanup, connected components, coloring, and CSV/JSON export run on the host CPU after the predictions are transferred back.


In [ ]:
import subprocess
import sys
from pathlib import Path

command = [
    sys.executable,
    "/kaggle/working/MedSAM3/count_cell_instances_tpu.py",
    "--config", str(CONFIG_PATH),
    "--weights", str(WEIGHTS_PATH),
    "--image", str(IMAGE_PATH),
    "--prompt", PROMPT,
    "--threshold", str(THRESHOLD),
    "--nms-iou", str(NMS_IOU),
    "--mask-nms-iou", str(MASK_NMS_IOU),
    "--min-area", str(MIN_AREA),
    "--max-hole-area", str(MAX_HOLE_AREA),
    "--resolution", str(RESOLUTION),
    "--output-dir", str(OUTPUT_DIR),
]

print("Running:\n", " ".join(command))
subprocess.run(command, check=True)


## 5. Display the colored instances, overlay, and count

In [ ]:
import json
import matplotlib.pyplot as plt
from PIL import Image

colored_path = OUTPUT_DIR / "instances_colored.png"
overlay_path = OUTPUT_DIR / "instances_overlay.png"
records_path = OUTPUT_DIR / "instances.json"

with records_path.open("r", encoding="utf-8") as handle:
    results = json.load(handle)

print("Cancer-cell instances counted:", results["count"])

colored = Image.open(colored_path)
overlay = Image.open(overlay_path)

plt.figure(figsize=(14, 10))
plt.imshow(colored)
plt.title(f"Colored cell instances — count: {results['count']}")
plt.axis("off")
plt.show()

plt.figure(figsize=(14, 10))
plt.imshow(overlay)
plt.title(f"MedSAM3 overlay — count: {results['count']}")
plt.axis("off")
plt.show()


In [ ]:
# Show the first measurements and all generated files.
import pandas as pd

measurements = pd.read_csv(OUTPUT_DIR / "instances.csv")
display(measurements.head(20))

print("\nGenerated files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f" - {path.name}: {path.stat().st_size:,} bytes")


## 6. Package the outputs for download

In [ ]:
import shutil

archive_path = shutil.make_archive(
    "/kaggle/working/medsam3_cell_count_results",
    "zip",
    root_dir=OUTPUT_DIR,
)
print("Created:", archive_path)


## Troubleshooting

### `torch` / `torch_xla` binary mismatch
Restart the Kaggle session after installation and rerun from the TPU import-verification cell.

### TPU is not detected
Confirm that the notebook accelerator is set to **TPU VM v5e-8**, then restart the session.

### Out-of-memory error
Try a smaller `RESOLUTION`, such as `768`. Keep the same resolution for repeated images so XLA can reuse the compiled graph.

### Too many tiny detections
Increase `MIN_AREA`, for example from `80` to `150`, and/or raise `THRESHOLD`.

### Missing cells
Lower `THRESHOLD` gradually, for example from `0.45` to `0.35`. Validate the result against manually counted images.

### First run is slow
That is expected: XLA compiles the model graph on the first inference for a new tensor shape.
